# Ingest Fact Data into Bronze Layer
Load all daily order CSVs from the landing zone into a single Bronze Delta table.
All columns ingested as strings/raw types to preserve original data exactly.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pyspark.sql.functions as F

catalog_name = 'quickbite'

In [0]:
order_schema = StructType([
    StructField("dt",                  StringType(),  True),
    StructField("order_ts",            StringType(),  True),
    StructField("order_id",            StringType(),  False),
    StructField("item_seq",            StringType(),  True),
    StructField("customer_id",         StringType(),  True),
    StructField("restaurant_id",       StringType(),  True),
    StructField("agent_id",            StringType(),  True),   # nullable — null for cancelled orders
    StructField("item_id",             StringType(),  True),
    StructField("quantity",            StringType(),  True),   # string — may contain typos
    StructField("item_price",          StringType(),  True),   # string — may have ₹ prefix
    StructField("discount_pct",        StringType(),  True),
    StructField("delivery_fee",        StringType(),  True),
    StructField("delivery_time_mins",  StringType(),  True),   # string — may be negative
    StructField("payment_mode",        StringType(),  True),
    StructField("order_status",        StringType(),  True),
    StructField("city",                StringType(),  True),
])

In [0]:
raw_path = "/Volumes/quickbite/source_data/raw/orders/landing/*.csv"

df = (spark.read
      .option("header", "true")
      .option("delimiter", ",")
      .schema(order_schema)
      .csv(raw_path)
      .withColumn("file_name",        F.col("_metadata.file_path"))
      .withColumn("ingest_timestamp", F.current_timestamp()))

print(f"Total rows ingested: {df.count()}")
display(df.limit(5))

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_orders")

print("brz_orders written:", spark.table(f"{catalog_name}.bronze.brz_orders").count(), "rows")